# DeepFaceLab Training (Colab GPU)
This notebook follows: collect face images -> train model -> export DFM.
Output target: `trained_model.dfm`.

Use the config cell to set your ZIP names and model options before running training.

In [ ]:
!nvidia-smi

In [ ]:
%cd /content\n
!git clone https://github.com/iperov/DeepFaceLab.git\n
%cd /content/DeepFaceLab\n
!python -m pip install -U pip\n
!python -m pip install -r requirements-colab.txt

## Upload datasets and configure training
Prepare and upload:
- `data_src.zip` (source face frames)
- `data_dst.zip` (target face frames)

The next config cell controls dataset names, model type, and output filename.

In [ ]:
import os
import shutil
from pathlib import Path

SRC_ZIP = "data_src.zip"
DST_ZIP = "data_dst.zip"
MODEL_NAME = "SAEHD"          # Options commonly used: SAEHD
OUTPUT_DFM = "trained_model.dfm"

WORKSPACE_DIR = Path("/content/DeepFaceLab/workspace")
DATA_SRC_DIR = WORKSPACE_DIR / "data_src"
DATA_DST_DIR = WORKSPACE_DIR / "data_dst"
MODEL_DIR = WORKSPACE_DIR / "model"

for directory in [DATA_SRC_DIR, DATA_DST_DIR, MODEL_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print("Config loaded")
print(f"SRC_ZIP={SRC_ZIP}")
print(f"DST_ZIP={DST_ZIP}")
print(f"MODEL_NAME={MODEL_NAME}")
print(f"OUTPUT_DFM={OUTPUT_DFM}")

In [ ]:
from google.colab import files

print("Upload your dataset ZIP files now (for example data_src.zip and data_dst.zip)")
uploaded = files.upload()
print(f"Uploaded files: {list(uploaded.keys())}")

In [ ]:
%cd /content/DeepFaceLab

!rm -rf workspace/data_src workspace/data_dst
!mkdir -p workspace/data_src workspace/data_dst

!unzip -o {SRC_ZIP} -d workspace/
!unzip -o {DST_ZIP} -d workspace/

!python main.py extract --input-dir workspace/data_src --output-dir workspace/data_src/aligned --detector s3fd
!python main.py extract --input-dir workspace/data_dst --output-dir workspace/data_dst/aligned --detector s3fd

In [ ]:
%cd /content/DeepFaceLab

!python main.py train --training-data-src-dir workspace/data_src/aligned --training-data-dst-dir workspace/data_dst/aligned --model-dir workspace/model --model {MODEL_NAME}

In [ ]:
%cd /content/DeepFaceLab

!python main.py exportdfm --model-dir workspace/model --model {MODEL_NAME} --output-file workspace/{OUTPUT_DFM}

In [ ]:
from google.colab import files

output_path = f'/content/DeepFaceLab/workspace/{OUTPUT_DFM}'
print(f'Downloading: {output_path}')
files.download(output_path)